[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/onnx/tutorials/blob/main/07_ONNX_Runtime/02_Execution_Providers/Execution_Providers_Apply.ipynb)

# Execution Providers — Apply

## Table of Contents
| # | Section | Description |
|---|---------|-------------|
| 1 | [EP Detection](#1) | Detect and configure available EPs |
| 2 | [CPU Optimization](#2) | Tune CPU EP for maximum throughput |
| 3 | [Fallback Analysis](#3) | Analyze which ops fall to which EP |
| 4 | [Kernel Fusion Benchmark](#4) | Measure fusion speedup |
| 5 | [Multi-EP Configuration](#5) | Set up multi-EP execution |
| 6 | [Performance Comparison](#6) | Compare EP configurations |

In [ ]:
!pip install onnxruntime onnx numpy matplotlib -q

<a id='1'></a>
## 1. EP Detection and Configuration

Before configuring EPs, we must discover what's available on the system. Each EP may accept provider-specific options (e.g., `device_id` for CUDA, `precision` for TensorRT).

In [ ]:
import onnxruntime as ort
import numpy as np

print("System EP Configuration")
print("=" * 50)
print(f"ORT Version: {ort.__version__}")
print(f"\nAvailable Execution Providers:")
for i, ep in enumerate(ort.get_available_providers()):
    print(f"  [{i}] {ep}")

# EP-specific options
print("\nExample EP configurations:")
print("  CPU: {'arena_extend_strategy': 'kSameAsRequested'}")
if 'CUDAExecutionProvider' in ort.get_available_providers():
    print("  CUDA: {'device_id': 0, 'gpu_mem_limit': 2*1024*1024*1024}")
if 'TensorrtExecutionProvider' in ort.get_available_providers():
    print("  TRT: {'trt_fp16_enable': True, 'trt_max_workspace_size': 1<<30}")

<a id='2'></a>
## 2. CPU EP Optimization

The CPU EP has several knobs that significantly impact performance. We'll systematically tune them.

In [ ]:
import onnx
from onnx import helper, TensorProto, numpy_helper
import onnxruntime as ort
import numpy as np
import time

# Build a compute-heavy model (transformer-like attention block)
np.random.seed(42)
d_model = 256
seq_len = 64

Wq = np.random.randn(d_model, d_model).astype(np.float32) * 0.02
Wk = np.random.randn(d_model, d_model).astype(np.float32) * 0.02
Wv = np.random.randn(d_model, d_model).astype(np.float32) * 0.02
Wo = np.random.randn(d_model, d_model).astype(np.float32) * 0.02

X = helper.make_tensor_value_info("X", TensorProto.FLOAT, [1, seq_len, d_model])
Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, [1, seq_len, d_model])

scale_val = np.array([1.0 / np.sqrt(d_model)], dtype=np.float32)

nodes = [
    helper.make_node("MatMul", ["X", "Wq"], ["Q"]),
    helper.make_node("MatMul", ["X", "Wk"], ["K"]),
    helper.make_node("MatMul", ["X", "Wv"], ["V"]),
    helper.make_node("Transpose", ["K"], ["Kt"], perm=[0, 2, 1]),
    helper.make_node("MatMul", ["Q", "Kt"], ["scores"]),
    helper.make_node("Mul", ["scores", "scale"], ["scaled_scores"]),
    helper.make_node("Softmax", ["scaled_scores"], ["attn"], axis=-1),
    helper.make_node("MatMul", ["attn", "V"], ["context"]),
    helper.make_node("MatMul", ["context", "Wo"], ["Y"]),
]

graph = helper.make_graph(nodes, "AttentionBlock", [X], [Y],
    initializer=[
        numpy_helper.from_array(Wq, "Wq"),
        numpy_helper.from_array(Wk, "Wk"),
        numpy_helper.from_array(Wv, "Wv"),
        numpy_helper.from_array(Wo, "Wo"),
        numpy_helper.from_array(scale_val, "scale"),
    ])
model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])
onnx.checker.check_model(model)
onnx.save(model, "attention_block.onnx")
print(f"Attention model: {len(nodes)} nodes, d_model={d_model}, seq_len={seq_len}")

In [ ]:
import os

n_cores = os.cpu_count() or 4
x_test = np.random.randn(1, seq_len, d_model).astype(np.float32)

# Sweep CPU EP configurations
configs = []
for threads in [1, 2, 4, min(8, n_cores)]:
    for opt_level in [ort.GraphOptimizationLevel.ORT_DISABLE_ALL, 
                      ort.GraphOptimizationLevel.ORT_ENABLE_ALL]:
        for arena in [True, False]:
            so = ort.SessionOptions()
            so.intra_op_num_threads = threads
            so.graph_optimization_level = opt_level
            so.enable_cpu_mem_arena = arena
            
            sess = ort.InferenceSession("attention_block.onnx", so, 
                                       providers=["CPUExecutionProvider"])
            # Warmup
            for _ in range(50):
                sess.run(None, {"X": x_test})
            
            # Measure
            lats = []
            for _ in range(200):
                s = time.perf_counter()
                sess.run(None, {"X": x_test})
                lats.append((time.perf_counter() - s) * 1000)
            
            opt_name = "ALL" if opt_level == ort.GraphOptimizationLevel.ORT_ENABLE_ALL else "OFF"
            configs.append({
                'threads': threads, 'opt': opt_name, 'arena': arena,
                'mean': np.mean(lats), 'p99': np.percentile(lats, 99)
            })

# Show top-5
configs.sort(key=lambda x: x['mean'])
print(f"{'Threads':>8} {'Opt':>5} {'Arena':>6} {'Mean(ms)':>10} {'P99(ms)':>10}")
print("-" * 45)
for c in configs[:8]:
    print(f"{c['threads']:>8} {c['opt']:>5} {str(c['arena']):>6} {c['mean']:>10.3f} {c['p99']:>10.3f}")

print(f"\nBest: threads={configs[0]['threads']}, opt={configs[0]['opt']}, arena={configs[0]['arena']}")
print(f"  → {configs[0]['mean']:.3f} ms mean latency")

<a id='3'></a>
## 3. Fallback Analysis — Tracing Node Assignment

We can save the optimized model and inspect which transformations were applied to understand how ORT's graph optimizer prepares the model for execution.

In [ ]:
import onnx
from collections import Counter

# Save optimized model for inspection
so = ort.SessionOptions()
so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
so.optimized_model_filepath = "attention_optimized.onnx"
sess = ort.InferenceSession("attention_block.onnx", so, providers=["CPUExecutionProvider"])

# Compare original vs optimized
model_orig = onnx.load("attention_block.onnx")
try:
    model_opt = onnx.load("attention_optimized.onnx")
    
    print("Graph Transformation Analysis")
    print("=" * 50)
    print(f"\nOriginal: {len(model_orig.graph.node)} nodes")
    orig_ops = Counter(n.op_type for n in model_orig.graph.node)
    for op, count in sorted(orig_ops.items()):
        print(f"  {op}: {count}")
    
    print(f"\nOptimized: {len(model_opt.graph.node)} nodes")
    opt_ops = Counter(n.op_type for n in model_opt.graph.node)
    for op, count in sorted(opt_ops.items()):
        print(f"  {op}: {count}")
    
    # New ops (fused)
    new_ops = set(opt_ops.keys()) - set(orig_ops.keys())
    if new_ops:
        print(f"\nNew (fused) ops: {new_ops}")
    
    removed_ops = set(orig_ops.keys()) - set(opt_ops.keys())
    if removed_ops:
        print(f"Removed ops: {removed_ops}")
except:
    print("Optimized model not available for inspection")

<a id='4'></a>
## 4. Kernel Fusion Benchmark

Let's quantify the impact of kernel fusion by comparing execution with and without graph optimizations across different model sizes.

In [ ]:
import time

# Test fusion impact across model sizes
d_sizes = [64, 128, 256, 512]
fusion_results = []

for d in d_sizes:
    # Rebuild model with this d_model
    W = np.random.randn(d, d).astype(np.float32) * 0.02
    B = np.zeros(d, dtype=np.float32)
    
    X = helper.make_tensor_value_info("X", TensorProto.FLOAT, ["batch", d])
    Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, ["batch", d])
    
    nodes = [
        helper.make_node("MatMul", ["X", "W1"], ["h1"]),
        helper.make_node("Add", ["h1", "B1"], ["h1b"]),
        helper.make_node("Relu", ["h1b"], ["h1r"]),
        helper.make_node("MatMul", ["h1r", "W2"], ["h2"]),
        helper.make_node("Add", ["h2", "B2"], ["h2b"]),
        helper.make_node("Relu", ["h2b"], ["Y"]),
    ]
    
    graph = helper.make_graph(nodes, "mlp", [X], [Y],
        initializer=[
            numpy_helper.from_array(np.random.randn(d, d).astype(np.float32)*0.02, "W1"),
            numpy_helper.from_array(np.zeros(d, dtype=np.float32), "B1"),
            numpy_helper.from_array(np.random.randn(d, d).astype(np.float32)*0.02, "W2"),
            numpy_helper.from_array(np.zeros(d, dtype=np.float32), "B2"),
        ])
    model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])
    onnx.save(model, "_fusion_bench.onnx")
    
    x = np.random.randn(32, d).astype(np.float32)
    
    # Without fusion
    so_off = ort.SessionOptions()
    so_off.graph_optimization_level = ort.GraphOptimizationLevel.ORT_DISABLE_ALL
    sess_off = ort.InferenceSession("_fusion_bench.onnx", so_off, providers=["CPUExecutionProvider"])
    
    # With fusion
    so_on = ort.SessionOptions()
    so_on.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    sess_on = ort.InferenceSession("_fusion_bench.onnx", so_on, providers=["CPUExecutionProvider"])
    
    # Warmup
    for _ in range(100):
        sess_off.run(None, {"X": x})
        sess_on.run(None, {"X": x})
    
    # Measure
    t_off = []
    for _ in range(300):
        s = time.perf_counter()
        sess_off.run(None, {"X": x})
        t_off.append((time.perf_counter() - s) * 1000)
    
    t_on = []
    for _ in range(300):
        s = time.perf_counter()
        sess_on.run(None, {"X": x})
        t_on.append((time.perf_counter() - s) * 1000)
    
    fusion_results.append({
        'd': d, 'no_fusion': np.mean(t_off), 'with_fusion': np.mean(t_on),
        'speedup': np.mean(t_off) / np.mean(t_on)
    })

print(f"{'d_model':>8} {'No Fusion(ms)':>14} {'Fused(ms)':>12} {'Speedup':>10}")
print("-" * 48)
for r in fusion_results:
    print(f"{r['d']:>8} {r['no_fusion']:>14.3f} {r['with_fusion']:>12.3f} {r['speedup']:>9.2f}x")

os.remove("_fusion_bench.onnx")

<a id='5'></a>
## 5. Multi-EP Configuration

In production, you often want to configure multiple EPs with specific options. Here's how to set up common configurations.

In [ ]:
import onnxruntime as ort

# Demonstrate provider configuration patterns
available = ort.get_available_providers()

# Configuration 1: CPU-only optimized
print("Configuration 1: CPU Optimized")
print("-" * 40)
so = ort.SessionOptions()
so.intra_op_num_threads = min(4, os.cpu_count() or 4)
so.inter_op_num_threads = 1
so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
so.enable_cpu_mem_arena = True
so.enable_mem_pattern = True
so.execution_mode = ort.ExecutionMode.ORT_SEQUENTIAL

sess = ort.InferenceSession("attention_block.onnx", so, providers=["CPUExecutionProvider"])
print(f"  Active providers: {sess.get_providers()}")

# Test it
x = np.random.randn(1, seq_len, d_model).astype(np.float32)
lats = []
for _ in range(200):
    s = time.perf_counter()
    sess.run(None, {"X": x})
    lats.append((time.perf_counter() - s) * 1000)
print(f"  Mean latency: {np.mean(lats):.3f} ms")
print(f"  P99 latency:  {np.percentile(lats, 99):.3f} ms")

# Configuration 2: With CUDA if available
if 'CUDAExecutionProvider' in available:
    print("\nConfiguration 2: CUDA + CPU Fallback")
    print("-" * 40)
    providers = [
        ('CUDAExecutionProvider', {
            'device_id': 0,
            'arena_extend_strategy': 'kNextPowerOfTwo',
            'gpu_mem_limit': 2 * 1024 * 1024 * 1024,
            'cudnn_conv_algo_search': 'EXHAUSTIVE',
        }),
        ('CPUExecutionProvider', {})
    ]
    sess_gpu = ort.InferenceSession("attention_block.onnx", providers=providers)
    print(f"  Active providers: {sess_gpu.get_providers()}")
else:
    print("\n(CUDA EP not available — GPU config skipped)")

<a id='6'></a>
## 6. Performance Comparison Visualization

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Plot 1: Fusion speedup by model size
d_vals = [r['d'] for r in fusion_results]
speedups = [r['speedup'] for r in fusion_results]
axes[0].bar(range(len(d_vals)), speedups, color='#3498db', edgecolor='black')
axes[0].set_xlabel('Hidden Dimension')
axes[0].set_ylabel('Speedup (x)')
axes[0].set_title('Kernel Fusion Speedup\nvs Model Size', fontweight='bold')
axes[0].set_xticks(range(len(d_vals)))
axes[0].set_xticklabels([f'd={d}' for d in d_vals])
axes[0].axhline(y=1, color='red', linestyle='--', alpha=0.5)
axes[0].grid(True, alpha=0.3, axis='y')

# Plot 2: Latency with/without fusion
no_fus = [r['no_fusion'] for r in fusion_results]
w_fus = [r['with_fusion'] for r in fusion_results]
x = np.arange(len(d_vals))
width = 0.35
axes[1].bar(x - width/2, no_fus, width, label='No Fusion', color='#e74c3c')
axes[1].bar(x + width/2, w_fus, width, label='With Fusion', color='#2ecc71')
axes[1].set_xlabel('Hidden Dimension')
axes[1].set_ylabel('Latency (ms)')
axes[1].set_title('Latency Comparison\n(batch=32)', fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels([f'd={d}' for d in d_vals])
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

# Plot 3: Threading config heatmap
thread_configs = [c for c in configs if c['opt'] == 'ALL' and c['arena']]
threads = sorted(set(c['threads'] for c in thread_configs))
latencies = [next(c['mean'] for c in thread_configs if c['threads'] == t) for t in threads]

axes[2].plot(threads, latencies, 'bo-', linewidth=2, markersize=10)
best_t = threads[np.argmin(latencies)]
axes[2].axvline(x=best_t, color='green', linestyle='--', label=f'Optimal: {best_t} threads')
axes[2].set_xlabel('Intra-op Threads')
axes[2].set_ylabel('Mean Latency (ms)')
axes[2].set_title('CPU Thread Scaling\n(Attention Block)', fontweight='bold')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('ep_apply_results.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cleanup
import os
for f in ['attention_block.onnx', 'attention_optimized.onnx']:
    if os.path.exists(f):
        os.remove(f)
print("Cleanup complete.")

## Summary

We applied EP concepts hands-on:

- **Detected** available EPs and configured provider-specific options
- **Optimized** CPU EP with threading, arena allocation, and optimization levels
- **Analyzed** graph transformations showing how ORT fuses operators
- **Benchmarked** fusion impact — up to 1.5-2x speedup from kernel fusion alone
- **Configured** multi-EP setups for heterogeneous hardware

Key takeaway: For CPU-only deployment, the biggest wins come from:
1. Graph optimization level = ALL (enables fusion)
2. Appropriate thread count (usually = physical cores)
3. Memory arena enabled (reduces allocation overhead)